In [ ]:
import pandas as pd
import numpy as np
import json

# ==============================
# 1. LOAD DATA
# ==============================
df = pd.read_csv('cleaned.csv')

print("Columns available:", df.columns.tolist())

# ==============================
# 2. VALIDATE REQUIRED COLUMNS
# ==============================
required_cols = ['crop_year', 'season', 'crop', 'yield_kg_ha']

# detect state column dynamically (since you broke it earlier)
state_col = [col for col in df.columns if 'state' in col]

if not state_col:
    raise ValueError("No state column found. Check cleaned.csv")
else:
    state_col = state_col[0]

required_cols.append(state_col)

missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# ==============================
# 3. YEAR NORMALIZATION
# ==============================
year_min = df['crop_year'].min()
year_max = df['crop_year'].max()

if year_max == year_min:
    raise ValueError("All crop_year values are same. Normalization will fail.")

df['year_normalized'] = (df['crop_year'] - year_min) / (year_max - year_min)

# ==============================
# 4. ONE-HOT ENCODING
# ==============================
df_encoded = pd.get_dummies(
    df,
    columns=[state_col, 'crop', 'season'],
    prefix=['state', 'crop', 'season']
)

# ==============================
# 5. SPLIT FEATURES & TARGET
# ==============================
y = df_encoded['yield_kg_ha']
X = df_encoded.drop(columns=['yield_kg_ha', 'crop_year'])

# ==============================
# 6. CREATE FEATURE CONTRACT
# ==============================
feature_cols = X.columns.tolist()

with open('feature_columns.json', 'w') as f:
    json.dump(feature_cols, f)

print(f"Contract Created → {len(feature_cols)} features")

# ==============================
# 7. SAVE FINAL DATA
# ==============================
df_final = pd.concat([X, y], axis=1)
df_final.to_csv('features.csv', index=False)

print("✅ features.csv saved")
print("✅ feature_columns.json saved")